# Question 7: Forecast August 2026 activity

County-level forecasts for households (the visits proxy), pounds and individuals, plus the graduate addendum forecasts for children, adults and seniors. The reproducible forecasting logic lives in `questions/q7_forecast_august/analysis.py`; this notebook runs it and explores its diagnostics and results.

> **Hard cutoff:** August 2026 data from any source is prohibited. The only permissible forecast input ends in July 2026. Do not search for or load August 2026 actuals.

## Setup and leakage guard

The notebook locates the repository root from the current directory, loads the canonical processed county-month panel, checks the hard cutoff, and then runs the same script used to generate the saved tables. Backtests use August 2023, 2024 and 2025, each using only observations strictly before its target month.

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src" / "qkit.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate repository root containing src/qkit.py")

sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import config as cfg
import qkit

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"Repository: {ROOT}")
print(f"Permitted training cutoff: {cfg.TRAIN_CUTOFF_YM}")
print(f"Forecast target (not observed): {cfg.FORECAST_TARGET_YM}")

In [ ]:
panel = qkit.county_month()
qkit.leakage_check(panel)
panel = panel.loc[panel["yearmonth"].le(cfg.TRAIN_CUTOFF_YM)].copy()
if panel["yearmonth"].max() != cfg.TRAIN_CUTOFF_YM:
    raise ValueError("Expected the permitted panel to end in July 2026")

display(pd.DataFrame({
    "rows": [len(panel)],
    "counties_with_reporting_history": [panel["fips"].nunique()],
    "months": [panel["yearmonth"].nunique()],
    "first_month": [panel["yearmonth"].min()],
    "last_permitted_month": [panel["yearmonth"].max()],
    "July 2026 reporting sites (statewide sum)": [panel.loc[panel.yearmonth.eq(cfg.TRAIN_CUTOFF_YM), "n_sites_reporting"].sum()],
}))

## Exploratory analysis: understand the panel before forecasting

Start by checking the panel's coverage and reporting completeness, then inspect statewide trends and county-level August/July variation. These are descriptive checks—not forecasts—and all calculations use the cutoff-filtered `panel` created above. No data from August 2026 or later is used.

In [ ]:
metrics = [*cfg.THREE_METRICS, *cfg.AGE_METRICS]

coverage = pd.DataFrame({
    "rows": [len(panel)],
    "distinct_counties": [panel.fips.nunique()],
    "distinct_months": [panel.yearmonth.nunique()],
    "missing_county_months_vs_87_counties": [87 * panel.yearmonth.nunique() - len(panel)],
    "first_yearmonth": [panel.yearmonth.min()],
    "last_yearmonth": [panel.yearmonth.max()],
})
quality = pd.DataFrame({
    "missing_values": panel[metrics + ["n_sites_reporting"]].isna().sum(),
    "zero_values": panel[metrics + ["n_sites_reporting"]].eq(0).sum(),
    "minimum": panel[metrics + ["n_sites_reporting"]].min(),
})
display(coverage)
display(quality)

# Counties absent from the activity panel are reporting gaps, not observed zeroes.
reference = qkit.mmg(years=[2024])[['fips', 'county_name']].drop_duplicates()
reference['fips'] = reference.fips.astype('string').str.zfill(5)
observed_fips = set(panel.fips.astype('string').str.zfill(5))
no_activity = reference.loc[~reference.fips.isin(observed_fips)]
display(no_activity)

In [ ]:
# Statewide totals by month, using county-month observations through July 2026 only.
state_monthly = (panel.groupby(["yearmonth", "date"], as_index=False)[metrics + ["n_sites_reporting"]]
                 .sum().sort_values("yearmonth"))
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for metric in cfg.THREE_METRICS:
    axes[0].plot(state_monthly.date, state_monthly[metric], label=metric.title(), linewidth=1.6)
axes[0].set(title="Statewide reported activity by month", ylabel="Reported monthly total")
axes[0].legend(ncol=3)
axes[0].grid(alpha=0.25)
axes[1].plot(state_monthly.date, state_monthly.n_sites_reporting, color="#8c564b", linewidth=1.6)
axes[1].set(title="Reporting-site count (summed across counties)", ylabel="Site-month reports", xlabel="Month")
axes[1].grid(alpha=0.25)
fig.tight_layout()
plt.show()

# Compare the final permitted month with recent reporting-site history.
latest_sites = state_monthly.loc[state_monthly.yearmonth.eq(cfg.TRAIN_CUTOFF_YM), "n_sites_reporting"].iloc[0]
trailing_sites = state_monthly.loc[state_monthly.yearmonth.between(202508, 202606), "n_sites_reporting"].mean()
print(f"July 2026 reporting sites: {latest_sites:,.0f}; previous 11-month average: {trailing_sites:,.1f}; difference: {100 * (latest_sites / trailing_sites - 1):+.1f}%")

In [ ]:
# County-level August/July ratios vary substantially; summarize by metric and year.
ratio_rows = []
for year in range(2022, 2026):
    july = panel.loc[panel.yearmonth.eq(year * 100 + 7), ["fips", *metrics]].set_index("fips")
    august = panel.loc[panel.yearmonth.eq(year * 100 + 8), ["fips", *metrics]].set_index("fips")
    ratios = august.div(july.where(july.gt(0))).replace([np.inf, -np.inf], np.nan)
    for metric in metrics:
        values = ratios[metric].dropna()
        ratio_rows.append({
            "year": year, "metric": metric, "counties": len(values),
            "median_ratio": values.median(),
            "q10_ratio": values.quantile(0.10),
            "q90_ratio": values.quantile(0.90),
        })
ratio_summary = pd.DataFrame(ratio_rows)
display(ratio_summary.loc[ratio_summary.metric.isin(cfg.THREE_METRICS)].round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, metric in zip(axes, cfg.THREE_METRICS):
    subset = ratio_summary.loc[ratio_summary.metric.eq(metric)]
    ax.errorbar(subset.year, subset.median_ratio,
                yerr=[subset.median_ratio - subset.q10_ratio,
                      subset.q90_ratio - subset.median_ratio],
                fmt="o-", capsize=4)
    ax.axhline(1, color="0.5", linestyle="--", linewidth=1)
    ax.set(title=metric.title(), xlabel="Year", xticks=range(2022, 2026))
    ax.grid(alpha=0.25)
axes[0].set_ylabel("August / July ratio (median; 10th–90th percentile across counties)")
fig.tight_layout()
plt.show()

## Historical seasonality and reporting composition

These descriptive checks use only the permitted panel. They help explain why the forecast should be treated as a forecast of **reported activity**: monthly totals can shift with both service volume and the number of sites reporting. August 2026 is not included in any input or calculation.

In [ ]:
forecast_metrics = [*cfg.THREE_METRICS, *cfg.AGE_METRICS]
state_month = panel.groupby(["year", "month"], as_index=False)[forecast_metrics + ["n_sites_reporting"]].sum()
seasonality = state_month.groupby("month")[forecast_metrics].mean()
seasonality = seasonality.div(seasonality.mean(axis=0), axis="columns")
seasonality.index.name = "calendar_month"
display(seasonality.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for metric in cfg.THREE_METRICS:
    axes[0].plot(seasonality.index, seasonality[metric], marker="o", label=metric.title())
axes[0].axhline(1, color="0.5", linestyle="--", linewidth=1)
axes[0].set(title="Statewide monthly seasonal index", xlabel="Month", ylabel="Index (1 = average)", xticks=range(1, 13))
axes[0].legend()

site_counts = state_month.groupby("year", as_index=False)["n_sites_reporting"].mean()
axes[1].plot(site_counts["year"], site_counts["n_sites_reporting"], marker="o", color="#8c564b")
axes[1].set(title="Average statewide reporting sites per month", xlabel="Year", ylabel="Sites reporting (monthly average)")
axes[1].grid(alpha=0.25)
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Run the leakage-safe backtests and forecast

The script compares seasonal naive, county August/July ratio, seasonal naive with drift, a ratio pooled toward the statewide median, and a per-reporting-site ratio. It selects separately by outcome using average county-level sMAPE over the three backtest years, while also reporting WAPE and MAE. The official competition score has not been confirmed, so sMAPE is a working selection metric—not a claim about the judges' metric.

In [ ]:
import runpy

analysis_path = ROOT / "questions" / "q7_forecast_august" / "analysis.py"
runpy.run_path(str(analysis_path), run_name="__main__")

In [ ]:
table_dir = ROOT / "outputs" / "tables"
forecasts = pd.read_csv(table_dir / "q7_forecasts.csv", dtype={"fips": "string"})
backtest_predictions = pd.read_csv(table_dir / "q7_backtest_predictions.csv", dtype={"fips": "string"})
backtest_scores = pd.read_csv(table_dir / "q7_backtest_scores.csv")
method_selection = pd.read_csv(table_dir / "q7_method_selection.csv")
uncertainty = pd.read_csv(table_dir / "q7_uncertainty_calibration.csv")

display(method_selection.loc[method_selection.selected_for_forecast].sort_values("metric").reset_index(drop=True))

## Backtest performance

The plot shows county-level sMAPE by backtest target. Lower is better. Compare fold-level performance as well as the average: there are only three backtest Augusts, so this is limited evidence, and a good average does not guarantee a good result in 2026.

In [ ]:
score_folds = backtest_scores.loc[backtest_scores["target_year"].ne("ALL")].copy()
score_folds["target_year"] = score_folds["target_year"].astype(int)
metrics_to_plot = [*cfg.THREE_METRICS, *cfg.AGE_METRICS]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, metric in zip(axes.flat, metrics_to_plot):
    subset = score_folds.loc[score_folds.metric.eq(metric)]
    for method, group in subset.groupby("method"):
        ax.plot(group.target_year, group.macro_smape_pct, marker="o", label=method.replace("_", " "))
    selected = method_selection.loc[method_selection.metric.eq(metric) & method_selection.selected_for_forecast, "method"]
    ax.set_title(f"{metric.title()} — selected: {selected.iloc[0] if len(selected) else 'n/a'}")
    ax.set_xticks([2023, 2024, 2025])
    ax.set_ylabel("County-level sMAPE (%)")
    ax.grid(alpha=0.25)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

display(
    score_folds.groupby(["metric", "method"], as_index=False)[["macro_smape_pct", "wape_pct", "mae"]]
    .mean().sort_values(["metric", "macro_smape_pct"]).round(2)
)

### Rubric-aligned performance: county RMSE by scored outcome

The draft graduate rubric scores the five required forecast outcomes using county-level RMSE. This chart recomputes RMSE from each selected method's August 2023, 2024, and 2025 backtest predictions. Each panel has its own units and scale, so compare years within an outcome—not the heights across different outcomes. Lower RMSE is better.

In [ ]:
rubric_metrics = [cfg.M_VISITS, cfg.M_POUNDS, *cfg.AGE_METRICS]
selected_methods = method_selection.loc[
    method_selection.selected_for_forecast, ["metric", "method"]
]
selected_backtest = backtest_predictions.merge(
    selected_methods, on=["metric", "method"], how="inner", validate="m:1"
).dropna(subset=["actual", "prediction"])
selected_backtest = selected_backtest.loc[
    selected_backtest.metric.isin(rubric_metrics)
].copy()
selected_backtest["squared_error"] = (
    selected_backtest.prediction - selected_backtest.actual
) ** 2
rmse_by_fold = (
    selected_backtest.groupby(["metric", "target_year"], as_index=False)
    .agg(counties=("fips", "nunique"), mse=("squared_error", "mean"))
)
rmse_by_fold["rmse"] = np.sqrt(rmse_by_fold.mse)
rmse_by_fold["metric_label"] = rmse_by_fold.metric.map({
    cfg.M_VISITS: "Households (visits proxy)",
    cfg.M_POUNDS: "Pounds",
    "children": "Children",
    "adults": "Adults",
    "seniors": "Seniors",
})
display(rmse_by_fold[["metric_label", "target_year", "counties", "rmse"]]
        .sort_values(["metric_label", "target_year"]).round(1))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, metric in zip(axes.flat, rubric_metrics):
    subset = rmse_by_fold.loc[rmse_by_fold.metric.eq(metric)].sort_values("target_year")
    ax.plot(subset.target_year, subset.rmse, marker="o", linewidth=2, color="#2171b5")
    ax.set_title(subset.metric_label.iloc[0])
    ax.set_xticks([2023, 2024, 2025])
    ax.set_ylabel("County RMSE (outcome units)")
    ax.grid(alpha=0.25)
axes.flat[-1].set_visible(False)
fig.suptitle("Selected-model backtest RMSE by outcome", fontsize=15)
fig.text(0.5, 0.01,
         "Each point is RMSE across counties for one held-out August. Units differ by panel; compare years within each panel only.",
         ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
plt.show()

## Final August 2026 prediction and empirical uncertainty range

This is the final point forecast from the selected methods. The statewide totals below sum the 86 counties with reporting history; Wilkin is not included because the panel has no activity history there. Its blank forecast must not be interpreted as zero. The county chart labels the ten largest predictions in each metric. The lower and upper columns are the 10th and 90th percentile empirical-error ranges from the selected method's backtests. They are **not calibrated guarantees**; only three target years are available.

In [ ]:
display(
    forecasts.groupby("metric", as_index=False).agg(
        counties_in_output=("fips", "nunique"),
        counties_forecasted=("prediction", "count"),
        august_2026_point_forecast_sum=("prediction", "sum"),
        selected_method=("selected_method", "first"),
    ).round(1).sort_values("metric")
)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, metric in zip(axes.flat, metrics_to_plot):
    subset = (forecasts.loc[forecasts.metric.eq(metric) & forecasts.prediction.notna()]
              .nlargest(10, "prediction").sort_values("prediction"))
    bars = ax.barh(subset.county_name, subset.prediction, color="#2171b5")
    ax.bar_label(bars, labels=[f"{value:,.0f}" for value in subset.prediction], padding=3, fontsize=7)
    ax.set_title(metric.title() + (" (visits proxy)" if metric == cfg.M_VISITS else ""))
    ax.set_xlabel("Predicted August 2026 value")
    ax.grid(axis="x", alpha=0.25)
    ax.ticklabel_format(axis="x", style="plain")
fig.suptitle("Ten largest county-level August 2026 point forecasts", fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, metric in zip(axes.flat, metrics_to_plot):
    subset = forecasts.loc[forecasts.metric.eq(metric) & forecasts.prediction.notna()].sort_values("prediction")
    x = np.arange(len(subset))
    ax.vlines(x, subset.lower_80, subset.upper_80, color="#9ecae1", linewidth=1)
    ax.scatter(x, subset.prediction, s=12, color="#2171b5")
    ax.set_title(metric.title())
    ax.set_xlabel("Counties ordered by point forecast")
    ax.set_ylabel("Forecast (whisker: empirical range)")
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

display(forecasts.loc[forecasts.forecast_status.eq("no_reporting_history"), ["fips", "county_name", "metric", "forecast_status"]])

## Graduate age-group checks

Individuals is derived county-by-county as children + adults + seniors. The table also flags forecasts whose age share falls outside that county's historical monthly range; these are reviewed rather than silently clipped. The range flag is a diagnostic, not a hard modeling constraint.

In [ ]:
age_forecasts = forecasts.loc[forecasts.metric.isin(cfg.AGE_METRICS)].copy()
age_share_flags = age_forecasts.loc[
    age_forecasts.age_share_within_historical_range.eq(False),
    ["fips", "county_name", "metric", "prediction", "age_share_forecast",
     "age_share_historical_min", "age_share_historical_max"],
].sort_values(["metric", "county_name"])
display(age_share_flags.round(3))

wide_forecast = forecasts.pivot(index="fips", columns="metric", values="prediction")
age_sum = wide_forecast[cfg.AGE_METRICS].sum(axis=1, min_count=3)
if not np.allclose(wide_forecast[cfg.M_INDIVIDUALS].dropna(), age_sum.dropna()):
    raise ValueError("Derived Individuals does not reconcile to the age-group forecasts")
print("Age-group sum check passed for every county with a forecast.")
print(f"Counties with at least one age-share range flag: {age_share_flags.fips.nunique()}")

## Interpretation and limitations

- Reported activity reflects both service demand and which organizations submitted data; July 2026's reporting-site count may still be incomplete.
- County-level data rolls up Hennepin and Ramsey to whole counties. It cannot produce split forecasts from this panel.
- Wilkin has no reporting history, so no statistical forecast is created for it.
- The selected pooled ratio is a simple baseline that won these three backtest folds; do not present this as proof it will be most accurate in August 2026.
- Confirm the competition score, county geography, July finality and Wilkin treatment with organizers. A person other than the model author must complete the repository-wide leakage audit before submission.

The saved submission-oriented table is `outputs/tables/q7_forecasts.csv`; detailed backtests and method comparisons are in the other `q7_*.csv` tables. Re-run this notebook after running the cells top to bottom to refresh all results.